In [ ]:
import boto3
import requests
import json

REGION = "us-east-1"
CLIENT_ID = "<YOUR_COGNITO_CLIENT_ID>"
USERNAME = "analyst_user"
PASSWORD = "SecurePassword123!"

# Authenticate against Cognito User Pool to obtain inbound authorization JWT
client = boto3.client('cognito-idp', region_name=REGION)
auth_response = client.initiate_auth(
    ClientId=CLIENT_ID,
    AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={'USERNAME': USERNAME, 'PASSWORD': PASSWORD}
)
access_token = auth_response['AuthenticationResult']['AccessToken']
print(f"Cognito Authentication Successful. Bearer Token acquired: {access_token[:30]}...")

In [ ]:
AGENTCORE_URL = "https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/<AGENT_ARN>/invocations"

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
    "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": "demo-session-12345678901234567890123"
}

def query_financial_agent(prompt: str):
    print(f"\n--- Query: {prompt} ---")
    response = requests.post(
        AGENTCORE_URL, 
        headers=headers, 
        json={"prompt": prompt}, 
        stream=True
    )
    
    # Process the Server-Sent Events (SSE) stream
    for line in response.iter_lines():
        if line:
            decoded_line = line.decode('utf-8')
            if decoded_line.startswith("data: "):
                data = json.loads(decoded_line[6:])
                print(data.get("event", ""), end="", flush=True)

In [ ]:
queries = [
    "What is the stock price for Amazon right now?",
    "What were the stock prices for Amazon in Q4 last year?",
    "Compare Amazon's recent stock performance to what analysts predicted in their reports.",
    "I’m researching AMZN give me the current price and any relevant information about their AI business.",
    "What is the total amount of office space Amazon owned in North America in 2024?"
]

for q in queries:
    query_financial_agent(q)

In [ ]:
# Verifying traces using Langfuse API
langfuse_auth = ('<LANGFUSE_PUBLIC_KEY>', '<LANGFUSE_SECRET_KEY>')
trace_response = requests.get(
    "https://cloud.langfuse.com/api/public/traces?sessionId=demo-session-12345678901234567890123",
    auth=langfuse_auth
)

print("\n--- Langfuse Trace Data ---")
print(json.dumps(trace_response.json(), indent=2))